# Train, validation, and test data

A model is usually used more than once during development. Features, preprocessing, hyperparameters, thresholds, and model families are selected from experiments. The data split must keep those development decisions separate from the final performance estimate.


## The three roles

The training set is used to fit parameters. The validation set is used to compare choices made during development. The test set is held back until the procedure is fixed and provides one final estimate of the selected procedure.


In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(4)
n = 1_200
X = rng.normal(size=(n, 2))
probability = 1 / (1 + np.exp(-(1.2 * X[:, 0] - 0.8 * X[:, 1])))
y = rng.binomial(1, probability)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=11, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=11, stratify=y_temp)

clean_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000))
clean_model.fit(X_train, y_train)
print(f'clean validation AUC: {roc_auc_score(y_val, clean_model.predict_proba(X_val)[:, 1]):.3f}')
print(f'clean test AUC:       {roc_auc_score(y_test, clean_model.predict_proba(X_test)[:, 1]):.3f}')

# Deliberately invalid: this feature is a noisy copy of the outcome.
leaked_X = np.column_stack([X, y + rng.normal(scale=0.05, size=n)])
X_train_leak, X_holdout_leak, y_train_leak, y_holdout_leak = train_test_split(leaked_X, y, test_size=0.4, random_state=11, stratify=y)
leaked_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1_000))
leaked_model.fit(X_train_leak, y_train_leak)
leaked_auc = roc_auc_score(y_holdout_leak, leaked_model.predict_proba(X_holdout_leak)[:, 1])
print(f'leaked holdout AUC:   {leaked_auc:.3f}  <- invalid experiment')


clean validation AUC: 0.747
clean test AUC:       0.777
leaked holdout AUC:   1.000  <- invalid experiment


## A clean split and a leaked split

The first model fits its scaler and logistic regression using training data, then reports validation and test AUC. The second model receives a feature made from the label plus a small amount of noise before the split. Its high holdout AUC is not useful evidence: the feature contains information that would not exist when making a real prediction.


## Sources of leakage

Leakage is not limited to an explicit copy of the target. It can occur when a transformation is fitted before splitting, when future values enter a feature, or when the same person or document appears in multiple partitions. A useful check is to ask whether every feature could have been computed at the prediction time.
